# Réalisation d'un RAG pour faire de la recherche rapide dans un panel de documents

Installation de llama-index et des différentes dépendances dont on va avoir besoin.

In [ ]:
%pip install llama-index-readers-file pymupdf
%pip install llama-index-readers-docling
%pip install llama-index-vector-stores-postgres
%pip install llama-index-embeddings-huggingface
%pip install llama-index-llms-llama-cpp
%pip install llama-index-node-parser-docling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 72.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 MB 8.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.9 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.2.90-cp310-cp310-linux_x86_64.whl size=3398616 sha256=339afb7c13874fe0deee105cd8243f1d2999f6e2814020ae18f3d148b0a87e65
  Stored in directory: /root/.cache/pip/wheels/3d/67/02/f950031435db4a5a02e6269f6adb6703bf1631c3616380f3c6
Successfully built llama-cpp-python


## Partie 1 : création de ma base de connaissance

Etape 1 : Télécharge le document https://arxiv.org/pdf/2307.09288 et https://arxiv.org/pdf/2404.08940 et sauvegarde les dans un dossier nommé 'data'. Tu peux renommer le premier 'llama2.pdf' et le second 'rag_mistral.pdf'. Pour cela utilise un Wget User Agent.

In [ ]:
!wget --user-agent

Etape 2 : Récupération des urls des différents documents dans une liste. Dans un premier temps tu peux tester sur un unique document.

In [ ]:
import glob
import pathlib
urls = []

Etape 3 : Lit les fichiers à partir d'un reader. Tu peux par exemple utiliser PyMuPDFReader ou Docling. Attention Docling est beaucoup plus long mais permet de lire des pdf scannés.

In [ ]:
from pathlib import Path
from llama_index.readers.file import PyMuPDFReader
from llama_index.readers.docling import DoclingReader

In [ ]:
documents = []

Etape 4 : Découpage du texte en chunk, en stockant les numéros de pages

In [ ]:
#split chunk
from llama_index.core.node_parser import SentenceSplitter

text_chunks = []
pages = []

Etape 5 : Création de noeuds llama-index contenant le texte et en metadonnées (node.metadata) le numéro de pages et les autres metadonnées associées

In [ ]:
# create text chunk
from llama_index.core.schema import TextNode

nodes = []

Etape 6 : Vectorisation du texte des chunks, grâce à un modèle de HuggingFace ("BAAI/bge-small-en"), et stockage du résultat dans node.embedding de chaque noeud.

In [ ]:
# sentence transformers
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
for node in nodes:
  node.embedding =

Etape 5 : Sauvegarde des vecteurs dans une base Vecteur que l'on va définir. En effet vous allez devoir écrire la fonction query de notre VectorStore. Pour vous aider vous avez : la classe mère, les fonctions get, add et delete ainsi que la fonction get_top_k_embeddings à votre disposition.

In [ ]:
from llama_index.core.vector_stores.types import BasePydanticVectorStore
from llama_index.core.vector_stores import (
    VectorStoreQuery,
    VectorStoreQueryResult,
)
from typing import List, Any, Optional, Dict
from llama_index.core.schema import TextNode, BaseNode
import os


class BaseVectorStore(BasePydanticVectorStore):
    """Simple custom Vector Store.

    Stores documents in a simple in-memory dict.

    """
    stores_text: bool = True

    # Add a dummy client property to satisfy the abstract base class requirement
    @property
    def client(self):
        return None  # Or any suitable value

    def get(self, text_id: str) -> List[float]:
        """Get embedding."""
        pass

    def add(
        self,
        nodes: List[BaseNode],
    ) -> List[str]:
        """Add nodes to index."""
        pass

    def delete(self, ref_doc_id: str, **delete_kwargs: Any) -> None:
        """
        Delete nodes using with ref_doc_id.

        Args:
            ref_doc_id (str): The doc_id of the document to delete.

        """
        pass

    def query(
        self,
        query: VectorStoreQuery,
        **kwargs: Any,
    ) -> VectorStoreQueryResult:
        """Get nodes for response."""
        pass

    # def persist(self, persist_path, fs=None) -> None:
    #     """Persist the SimpleVectorStore to a directory.

    #     NOTE: we are not implementing this for now.

    #     """
    #     pass

In [ ]:
from typing import Tuple
import numpy as np


def get_top_k_embeddings(
    query_embedding: List[float],
    doc_embeddings: List[List[float]],
    doc_ids: List[str],
    similarity_top_k: int = 5,
) -> Tuple[List[float], List]:
    """Get top nodes by similarity to the query."""
    # dimensions: D
    qembed_np = np.array(query_embedding)
    # dimensions: N x D
    dembed_np = np.array(doc_embeddings)
    # dimensions: N
    dproduct_arr = np.dot(dembed_np, qembed_np)
    # dimensions: N
    norm_arr = np.linalg.norm(qembed_np) * np.linalg.norm(
        dembed_np, axis=1, keepdims=False
    )
    # dimensions: N
    cos_sim_arr = dproduct_arr / norm_arr

    # now we have the N cosine similarities for each document
    # sort by top k cosine similarity, and return ids
    tups = [(cos_sim_arr[i], doc_ids[i]) for i in range(len(doc_ids))]
    sorted_tups = sorted(tups, key=lambda t: t[0], reverse=True)

    sorted_tups = sorted_tups[:similarity_top_k]

    result_similarities = [s for s, _ in sorted_tups]
    result_ids = [n for _, n in sorted_tups]
    return result_similarities, result_ids

In [ ]:
from pydantic import Field
from typing import cast
class VectorStoreRAG(BaseVectorStore):
    """VectorStoreRAG (add/get/delete already implemented)."""

    stores_text: bool = True
    node_dict: Dict[str, BaseNode] = Field(default_factory=dict)

    def get(self, text_id: str) -> List[float]:
        """Get embedding."""
        return self.node_dict[text_id]

    def add(
        self,
        nodes: List[BaseNode],
    ) -> List[str]:
        """Add nodes to index."""
        for node in nodes:
            self.node_dict[node.node_id] = node

    def delete(self, node_id: str, **delete_kwargs: Any) -> None:
        """
        Delete nodes using with node_id.

        Args:
            node_id: str

        """
        del self.node_dict[node_id]

    def query(
        self,
        query: VectorStoreQuery,
        **kwargs: Any,
    ) -> VectorStoreQueryResult:
        """Get nodes for response."""

        query_embedding = query.query_embedding
        doc_embeddings = [n.embedding for n in self.node_dict.values()]
        doc_ids = [n.node_id for n in self.node_dict.values()]

        result_similarities, result_ids = get_top_k_embeddings(query_embedding, doc_embeddings, doc_ids, query.similarity_top_k)

        results_nodes = [self.node_dict[node_id] for node_id in result_ids]

        return VectorStoreQueryResult(nodes=results_nodes, similarities=result_similarities, ids=result_ids)

In [ ]:
vector_store = VectorStoreRAG()
vector_store.add(nodes)

Etape 6 : Vérifie que vector_store contient bien ton premier noeud

## Partie 2 : en fonction de la question recherche et génération de la réponse

Etape 1 : Pose une question

In [ ]:
query_str = "Can you tell me about the key concepts for safety finetuning"

Etape 2 : Vectorisation de la question

In [ ]:
query_embedding =

Etape 3 : Récupération des 2 passages les plus pertinents

In [ ]:
# construct vector store query
from llama_index.core.vector_stores import VectorStoreQuery
query_obj = VectorStoreQuery(query_embedding=query_embedding, similarity_top_k=2)
query_result = vector_store.query(query_obj)

for similarity, node in zip(query_result.similarities, query_result.nodes):
    print(
        "\n----------------\n"
        f"[Node ID {node.node_id}] Similarity: {similarity}\n\n"
        f"{node.get_content(metadata_mode='all')}"
        "\n----------------\n\n"
    )

Etape 4 : Récupére les scores et associe les au noeud avec NodeWithSCore et stoque les dans une liste

In [ ]:
from llama_index.core.schema import NodeWithScore
from typing import Optional

nodes_with_scores = [NodeWithScore(node=node, score=similarity) for similarity, node in zip(query_result.similarities, query_result.nodes)]

In [ ]:
print([node.score for node in nodes_with_scores])

Etape 5 : Creation d'un classe qui reprend les étapes précédentes et à partir d'une question revoie directement la liste des NodeWithScore. On peut reprendre tout ce qu'on a fait précédemment et notamment VectorStoreRAG.

In [ ]:
from llama_index.core import QueryBundle
from llama_index.core.retrievers import BaseRetriever
from typing import Any, List


class VectorDBRetriever(BaseRetriever):
    """Retriever over a postgres vector store."""

    def __init__(
        self,
        vector_store: VectorStoreRAG,
        embed_model: Any,
        similarity_top_k: int = 2,
    ) -> None:
        """Init params."""
        self._vector_store = vector_store
        self._embed_model = embed_model
        self._similarity_top_k = similarity_top_k
        super().__init__()

    def _retrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
        """Retrieve."""

        query_embedding = embed_model.get_query_embedding(
            query_bundle.query_str
        )
        vector_store_query = VectorStoreQuery(
            query_embedding=query_embedding,
            similarity_top_k=self._similarity_top_k,
        )
        query_result = vector_store.query(vector_store_query)

        nodes_with_scores = [NodeWithScore(node=node, score=similarity) for similarity, node in zip(query_result.similarities, query_result.nodes)]

        return nodes_with_scores

Etape 6 : Initialisation du retriever avec la classe précédente

In [ ]:
retriever =

Etape 7 : Chargement du LLM Qwen2.5-0.5B-Instruct-GGUF de huggingface et plus précisément de qwen2.5-0.5b-instruct-q4_0.gguf.

In [ ]:
from llama_index.llms.llama_cpp import LlamaCPP

model_url = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/qwen2.5-0.5b-instruct-q4_0.gguf"

llm = LlamaCPP(
    # You can pass in the URL to a GGML model to download it automatically
    model_url=model_url,
    temperature=,
    max_new_tokens=,
    context_window=,
    model_kwargs={"n_gpu_layers": 1},
    verbose=True,
)

Etape 8 : Définit le query engine

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine.from_args(retriever, llm=llm)

Etape 8 : Visualise le prompt et modifie le.

In [ ]:
from llama_index.core import PromptTemplate
#visualise le

# nouveau prompt

# modifie le
query_engine.update_prompts()

Etape 6 : Obtient la réponse à la question ainsi que les passages qui ont permis d'y répondre.

In [ ]:
response = query_engine.query(query_str)

print(response)

# Finetuning d'un modèle pytorch

Etape 1 : Chargement du dataset "yelp_review_full"

In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
  Attempting uninstall: dill
    Found existing installation: dill 0.3.9
    Uninstalling dill-0.3.9:
      Successfully uninstalled dill-0.3.9
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.17
    Uninstalling multiprocess-0.70.17:
      Successfully uninstalled multiprocess-0.70.17
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed.

Etape 2 : Chargement du tokenizer "google-bert/bert-base-cased" et vectorisation du dataset.

Cette étape prend plusieurs minutes.

In [ ]:
from transformers import AutoTokenizer

tokenizer =

def tokenize_function(examples):
  return tokenizer()

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Etape 3 : Division du tokenized_datasets en un jeu d'entrainement et d'évaluation de 1000 exemples.

In [ ]:
small_train_dataset =
small_eval_dataset =

Etape 4 : Chargement du modèle de classification associé au tokenizer (num_labels=5).

In [ ]:
from transformers import AutoModelForSequenceClassification

model =

Etape 5 : Initialisation des arguments d'entrainement et notamment de sont dossier de sortie "train_output" avec une évaluation à la fin de chaque époque.

In [ ]:
from transformers import TrainingArguments

Etape 6 : Chargement de la metrique "accuracy" et création d'une fonction qui permet de carculer les metriques sur les prédictions.

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import evaluate

metric =

In [ ]:
def compute_metrics(eval_pred):
  return

Etape 7 : Initialisation du trainer et lancement de l'entrainement.

Pour que l'entrainement fonctionne il faut se créer un compte wandb. Vous n'êtes pas obligés de lancer l'entrainement car celui-ci est long et est juste un exemple.

In [ ]:
from transformers import TrainingArguments, Trainer

trainer =

# Définissez votre propre cas d'usage

Pensez à un cas d'usage que vous souhaitez résoudre puis concevez la solution.

* Quel type de modèle ?
* Quel type d'entrainement va être necessaire ?
* En fonction de votre cas d'usage trouvez un jeu de donnée sur huggingface ou un modèle pré-entrainé.